In [ ]:
import pandas as pd
import numpy as np # for random data generation
import matplotlib.pyplot as plt
import os
import torch
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler, LabelEncoder #import OneHotEncoder
from sklearn.model_selection import KFold ,StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from tqdm import tqdm
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from lightgbm import LGBMRegressor




In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv("/kaggle/input/q1-ka-ai-2026/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
df["Delivery_Time"].hist(bins=30, edgecolor='black')

plt.xlabel("Delivery_Time")
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean= df.drop(columns=['Order_ID'])
df_clean

In [ ]:
# Task 2: Write your code here:
the_null= df_clean.isnull().sum()

the_null[the_null>0]

In [ ]:
the_null[the_null>0].index.tolist()

In [ ]:
df_clean = df_clean.dropna(subset=['Delivery_Time'])

for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:

    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

df_clean.head()

In [ ]:
df_clean["Courier_Experience_yrs"] = df_clean["Courier_Experience_yrs"].fillna(df_clean["Courier_Experience_yrs"].mean())

In [ ]:
df_clean.info()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
le = LabelEncoder() # Instantiate LabelEncoder

for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:

    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
df_clean["Vehicle_Type"] = le.fit_transform(df_clean['Vehicle_Type'].astype(str))
df_clean.info()

In [ ]:
# Task 5: Write your code here:
features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scale = StandardScaler()
df_clean[features] = scale.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here:


In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time",axis=1)
y = df_clean['Delivery_Time']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
model=RandomForestRegressor(n_estimators=100)

In [ ]:
# Task 2,3,4,5: Write your code here:
lr_mae = []
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]




  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)
  lr_mae.append(mae)




In [ ]:
print(f"  Average MAE: {np.mean(lr_mae):.4f}")

In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs["tree"] = model.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
%pip install catboost

from catboost import CatBoostRegressor


In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_results= {}
for name in models:
  all_results[name] = {'mae': []}
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae =  mean_absolute_error(y_test, y_pred)
    all_results[model_name]["mae"].append(mae)

In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")